# Zomato Delivery Operations Analysis Using PySpark
## Stage 3 & 4: Exploratory Analysis, Spark SQL & Window Functions

**Author:** Computer Engineering Mini-Project  
**Framework:** Apache Spark via PySpark DataFrames & Spark SQL  
**Dataset:** `data/processed/zomato_cleaned.parquet` (45,584 enriched delivery records)

---

### Analytical Agenda:
1. **Analysis A:** Overall Delivery Performance & Duration Percentiles
2. **Analysis B:** Traffic Density Impact
3. **Analysis C:** Weather Conditions Impact
4. **Analysis D:** Fleet Vehicle Type & Condition Analysis
5. **Analysis E:** Delivery Partner Attributes (Age, Rating, Order Stacking)
6. **Analysis F:** Meal / Order Type Analysis
7. **Analysis G:** Festival Surges & Urban Category Hierarchy
8. **Analysis H:** Temporal Variations (Day of Week, Peak Hour)
9. **Analysis I:** Geospatial Straight-Line Distance vs. Delivery Time
10. **Stage 4:** Spark SQL Queries, Window Functions (Ranking, Rolling Avg), and Catalyst Plan (`explain()`)

In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from src.data_loader import get_spark_session
from src.analysis import (
    load_cleaned_data, analyze_overall_delivery_performance,
    analyze_traffic_impact, analyze_weather_impact,
    analyze_vehicle_performance, analyze_delivery_person_factors,
    analyze_order_type_impact, analyze_festival_and_city,
    analyze_temporal_patterns, analyze_geographic_distance,
    run_spark_sql_queries, demonstrate_window_functions, profile_execution_and_caching
)

spark = get_spark_session('Stage3_ExploratoryAnalysis')
df = load_cleaned_data(spark, '../data/processed/zomato_cleaned.parquet')
print(f'Loaded cleaned dataset with {df.count():,} rows and {len(df.columns)} columns.')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 23:47:37 WARN Utils: Your hostname, Rajdeeps-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/10/13 23:47:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 23:47:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loaded cleaned dataset with 45,584 rows and 31 columns.


### Analysis A: Overall Delivery Performance
**Research Question:** What is the baseline distribution, central tendency, and dispersion of delivery durations?

In [2]:
perf_stats = analyze_overall_delivery_performance(df)
for k, v in perf_stats.items():
    print(f'{k:<22}: {v}')

total_records         : 45584
mean_duration         : 26.29
stddev_duration       : 9.38
min_duration          : 10
max_duration          : 54
p10                   : 15.0
p25                   : 19.0
median_p50            : 26.0
p75                   : 32.0
p90                   : 39.0
p95                   : 44.0
p99                   : 49.0


### Analysis B: Traffic Congestion Impact
**Research Question:** Which traffic conditions are associated with longer delivery times?

In [3]:
traffic_df = analyze_traffic_impact(df)
traffic_df.show(truncate=False)

+--------------------+--------------+-----------------+--------------------+-----------------+--------------------+
|Road_traffic_density|delivery_count|avg_delivery_time|median_delivery_time|max_delivery_time|stddev_delivery_time|
+--------------------+--------------+-----------------+--------------------+-----------------+--------------------+
|Jam                 |14139         |31.18            |31                  |54               |9.94                |
|High                |4423          |27.24            |27                  |54               |8.4                 |
|Medium              |10945         |26.7             |26                  |49               |8.56                |
|Unknown             |601           |26.54            |26                  |54               |9.46                |
|Low                 |15476         |21.27            |20                  |44               |6.8                 |
+--------------------+--------------+-----------------+-----------------

### Analysis C: Weather Conditions Impact
**Research Question:** How do adverse atmospheric conditions relate to delivery duration?

In [4]:
weather_df = analyze_weather_impact(df)
weather_df.show(truncate=False)

+------------------+--------------+-----------------+--------------------+-----------------+-----------------+
|Weather_conditions|delivery_count|avg_delivery_time|median_delivery_time|min_delivery_time|max_delivery_time|
+------------------+--------------+-----------------+--------------------+-----------------+-----------------+
|Cloudy            |7533          |28.92            |28                  |10               |54               |
|Fog               |7653          |28.91            |28                  |10               |54               |
|Unknown           |616           |26.55            |26                  |10               |54               |
|Windy             |7422          |26.12            |26                  |10               |54               |
|Sandstorms        |7494          |25.88            |26                  |10               |54               |
|Stormy            |7584          |25.87            |26                  |10               |54               |
|

### Analysis D: Vehicle Type & Fleet Maintenance
**Research Question:** Does vehicle type or vehicle maintenance tier affect delivery performance?

In [5]:
by_type, by_cond = analyze_vehicle_performance(df)
print('--- By Vehicle Type ---')
by_type.show(truncate=False)
print('--- By Vehicle Maintenance Condition ---')
by_cond.show(truncate=False)

--- By Vehicle Type ---
+----------------+--------------+-----------------+--------------------+
|Type_of_vehicle |delivery_count|avg_delivery_time|median_delivery_time|
+----------------+--------------+-----------------+--------------------+
|motorcycle      |26429         |27.61            |26                  |
|bicycle         |68            |26.43            |26                  |
|scooter         |15273         |24.48            |24                  |
|electric_scooter|3814          |24.47            |24                  |
+----------------+--------------+-----------------+--------------------+

--- By Vehicle Maintenance Condition ---


+-----------------+--------------+-----------------+--------------------+
|Vehicle_condition|delivery_count|avg_delivery_time|median_delivery_time|
+-----------------+--------------+-----------------+--------------------+
|0                |15005         |30.07            |28                  |
|1                |15028         |24.35            |24                  |
|2                |15031         |24.45            |24                  |
|3                |520           |26.49            |26                  |
+-----------------+--------------+-----------------+--------------------+



### Analysis E: Delivery Partner Attributes (Age, Rating, Multiple Deliveries)
**Research Question:** Is delivery time correlated with courier ratings or stacked orders?

In [6]:
courier_analysis = analyze_delivery_person_factors(df)
print('Pearson Correlation Coefficients:')
for metric, corr in courier_analysis['correlations'].items():
    print(f'  {metric:<30}: {corr}')

print('\n--- Multiple Deliveries Stacking ---')
courier_analysis['multi_deliveries'].show(truncate=False)

print('\n--- Rating Bands ---')
courier_analysis['rating_bands'].show(truncate=False)

Pearson Correlation Coefficients:
  Age_vs_Time                   : 0.2929
  Rating_vs_Time                : -0.333
  MultipleDeliveries_vs_Time    : 0.3783

--- Multiple Deliveries Stacking ---
+-------------------+--------------+-----------------+--------------------+
|multiple_deliveries|delivery_count|avg_delivery_time|median_delivery_time|
+-------------------+--------------+-----------------+--------------------+
|0                  |14094         |22.88            |22                  |
|1                  |29144         |26.72            |26                  |
|2                  |1985          |40.45            |40                  |
|3                  |361           |47.82            |48                  |
+-------------------+--------------+-----------------+--------------------+


--- Rating Bands ---


+-------------------+--------------+-----------------+
|Rating_Band        |delivery_count|avg_delivery_time|
+-------------------+--------------+-----------------+
|< 4.0 (Low)        |1367          |36.83            |
|4.0 - 4.4 (Average)|6694          |34.65            |
|4.5 - 4.7 (Good)   |19288         |24.45            |
|4.8 - 5.0 (Top)    |18235         |24.39            |
+-------------------+--------------+-----------------+



### Analysis F: Order Type Impact
**Research Question:** Does the category of food ordered (Meal, Snack, Drinks, Buffet) correlate with delivery duration?

In [7]:
order_type_df = analyze_order_type_impact(df)
order_type_df.show(truncate=False)

+-------------+--------------+-----------------+--------------------+--------------------+
|Type_of_order|delivery_count|avg_delivery_time|median_delivery_time|stddev_delivery_time|
+-------------+--------------+-----------------+--------------------+--------------------+
|Meal         |11456         |26.42            |26                  |9.42                |
|Snack        |11530         |26.29            |26                  |9.4                 |
|Buffet       |11277         |26.28            |26                  |9.41                |
|Drinks       |11321         |26.19            |25                  |9.3                 |
+-------------+--------------+-----------------+--------------------+--------------------+



### Analysis G: Festival Surges & City Category
**Research Question:** How severely do festivals and urban density affect delivery latency?

In [8]:
by_festival, by_city = analyze_festival_and_city(df)
print('--- Festival Status ---')
by_festival.show(truncate=False)
print('--- City Urban Hierarchy ---')
by_city.show(truncate=False)

--- Festival Status ---


+--------+--------------+-----------------+--------------------+
|Festival|delivery_count|avg_delivery_time|median_delivery_time|
+--------+--------------+-----------------+--------------------+
|Yes     |896           |45.52            |45                  |
|No      |44460         |25.98            |25                  |
|Unknown |228           |11.17            |11                  |
+--------+--------------+-----------------+--------------------+

--- City Urban Hierarchy ---
+-------------+--------------+-----------------+--------------------+
|City         |delivery_count|avg_delivery_time|median_delivery_time|
+-------------+--------------+-----------------+--------------------+
|Semi-Urban   |164           |49.73            |49                  |
|Metropolitian|34087         |27.31            |26                  |
|Urban        |10133         |22.98            |22                  |
|Unknown      |1200          |22.06            |21                  |
+-------------+----------

### Analysis I: Straight-Line Haversine Distance vs. Delivery Time
**Research Question:** How strongly does estimated customer-restaurant distance correlate with duration?

In [9]:
dist_analysis = analyze_geographic_distance(df)
print(f'Pearson Correlation (Distance vs Delivery Time): {dist_analysis["correlation"]}')
dist_analysis['binned_dist'].show(truncate=False)

Pearson Correlation (Distance vs Delivery Time): 0.3205


+------------------+--------------+-----------------+--------------------+
|Distance_Bin      |delivery_count|avg_delivery_time|median_delivery_time|
+------------------+--------------+-----------------+--------------------+
|15 - 21 km (Far)  |8046          |29.86            |29                  |
|10 - 15 km (Long) |11293         |29.83            |29                  |
|5 - 10 km (Medium)|11305         |24.39            |25                  |
|< 5 km (Short)    |11300         |22.19            |21                  |
+------------------+--------------+-----------------+--------------------+



### Stage 4: Advanced Spark SQL Queries & Temporary Views
We demonstrate Spark SQL with ANSI SQL queries directly against our registered DataFrame view.

In [10]:
sql_results = run_spark_sql_queries(spark, df)
print('--- Query 1: Traffic by City Hierarchy ---')
sql_results['traffic_by_city'].show(10, truncate=False)

print('--- Query 2: Adverse Weather & Vehicle Condition Matrix ---')
sql_results['weather_vehicle_matrix'].show(10, truncate=False)

print('--- Query 3: Multi-Delivery Stacking during Festivals ---')
sql_results['festival_stacking'].show(10, truncate=False)

--- Query 1: Traffic by City Hierarchy ---
+-------------+--------------------+------------+----------------+---------------+
|City         |Road_traffic_density|order_volume|avg_duration_min|avg_distance_km|
+-------------+--------------------+------------+----------------+---------------+
|Metropolitian|Jam                 |11087       |31.91           |12.1           |
|Metropolitian|High                |3364        |28.11           |4.42           |
|Metropolitian|Unknown             |464         |27.72           |9.97           |
|Metropolitian|Medium              |8320        |27.61           |11.26          |
|Metropolitian|Low                 |10852       |22.13           |8.34           |
|Semi-Urban   |High                |17          |50.06           |4.9            |
|Semi-Urban   |Jam                 |135         |49.89           |13.84          |
|Semi-Urban   |Unknown             |1           |48.0            |6.21           |
|Semi-Urban   |Medium              |11      

+------------------+-----------------+------------+----------------+--------------------+
|Weather_conditions|Vehicle_condition|total_orders|avg_duration_min|avg_kitchen_prep_min|
+------------------+-----------------+------------+----------------+--------------------+
|Fog               |0                |2512        |31.88           |10.03               |
|Stormy            |0                |2482        |29.88           |9.96                |
|Sandstorms        |0                |2524        |29.87           |9.99                |
|Fog               |2                |2561        |27.53           |9.96                |
|Fog               |1                |2580        |27.4            |9.9                 |
|Sunny             |0                |2423        |26.91           |9.95                |
|Stormy            |1                |2525        |23.99           |9.99                |
|Sandstorms        |2                |2471        |23.94           |10.02               |
|Stormy   

### Stage 4: Window Functions
We implement two PySpark Window functions:
1. `DENSE_RANK()`: Ranking traffic congestion severity within each city category.
2. `ROLLING AVERAGE`: 7-day moving average of daily delivery times over ordered operational dates.

In [11]:
window_results = demonstrate_window_functions(df)
print('--- Window 1: Dense Rank of Traffic per City ---')
window_results['traffic_ranking_window'].show(15, truncate=False)

print('--- Window 2: 7-Day Rolling Average Daily Delivery Time ---')
window_results['rolling_daily_avg_window'].show(10, truncate=False)

--- Window 1: Dense Rank of Traffic per City ---


+-------------+--------------------+------------+---------------------+
|City         |Road_traffic_density|avg_duration|traffic_severity_rank|
+-------------+--------------------+------------+---------------------+
|Metropolitian|Jam                 |31.91       |1                    |
|Metropolitian|High                |28.11       |2                    |
|Metropolitian|Unknown             |27.72       |3                    |
|Metropolitian|Medium              |27.61       |4                    |
|Metropolitian|Low                 |22.13       |5                    |
|Semi-Urban   |High                |50.06       |1                    |
|Semi-Urban   |Jam                 |49.89       |2                    |
|Semi-Urban   |Unknown             |48.0        |3                    |
|Semi-Urban   |Medium              |47.45       |4                    |
|Urban        |Jam                 |27.74       |1                    |
|Urban        |High                |24.13       |2              

+----------+------------+--------------+----------------------+
|Order_Date|daily_orders|daily_avg_time|7_day_rolling_avg_time|
+----------+------------+--------------+----------------------+
|2022-02-11|970         |23.42         |23.42                 |
|2022-02-12|864         |30.06         |26.74                 |
|2022-02-13|957         |23.36         |25.61                 |
|2022-02-14|851         |30.06         |26.73                 |
|2022-02-15|945         |23.29         |26.04                 |
|2022-02-16|861         |30.3          |26.75                 |
|2022-02-17|939         |23.5          |26.28                 |
|2022-02-18|855         |29.67         |27.18                 |
|2022-03-01|1140        |23.23         |26.2                  |
|2022-03-02|1012        |29.51         |27.08                 |
+----------+------------+--------------+----------------------+
only showing top 10 rows


### Stage 4: Execution Plan Profiling and Caching Experiment

In [12]:
profile_execution_and_caching(df)
spark.stop()
print('SparkSession successfully stopped.')


--- [Spark Plan Profiling: explain()] ---


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[City#18], functions=[count(1)])
   +- Exchange hashpartitioning(City#18, 200), ENSURE_REQUIREMENTS, [plan_id=1397]
      +- HashAggregate(keys=[City#18], functions=[partial_count(1)])
         +- Project [City#18]
            +- Filter (isnotnull(Time_taken (min)#19) AND (Time_taken (min)#19 > 25))
               +- FileScan parquet [City#18,Time_taken (min)#19] Batched: true, DataFilters: [isnotnull(Time_taken (min)#19), (Time_taken (min)#19 > 25)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/rajdeepkadam/Desktop/Zomato Delivery Operations Analysis U..., PartitionFilters: [], PushedFilters: [IsNotNull(`Time_taken (min)`), GreaterThan(`Time_taken (min)`,25)], ReadSchema: struct<City:string,Time_taken (min):int>



--- [Caching Performance Experiment] ---


 Uncached Count Execution Time: 0.0810 seconds
 Cached Count Execution Time:   0.0519 seconds
 Cache Speedup Ratio:          1.56x


SparkSession successfully stopped.
